# AIC2024 Phase 1 - index tren Kaggle (corpus lon)

Pipeline: TransNetV2 -> shot -> keyframe -> SQLite/JPEG.

## Chuan bi
1. **Settings -> Accelerator -> GPU P100** (hoac T4 x2).
2. **Settings -> Internet -> On** (can de cai package / tai model).
3. Attach corpus video dang **Kaggle Dataset** vao `/kaggle/input/...` (co the attach nhieu batch cung luc).
4. Attach code repo dang Kaggle Dataset, hoac `git clone` vao `/kaggle/working/IR`.

## Luu y
- Session toi da 12h, GPU quota ~30h/tuan.
- `/kaggle/working` luu toi da ~20GB output; `/kaggle/temp` co nhieu scratch hon nhung **khong duoc luu**.
- Neu upload bi gioi han, chia corpus thanh nhieu dataset/batch: notebook tu quet tat ca video trong `/kaggle/input` va chia shard.

Notebook chia corpus thanh shard nho, dung lai artifact cu de resume qua nhieu session.

In [ ]:
!nvidia-smi
import shutil, sys, torch
print(f"python={sys.version.split()[0]} torch={torch.__version__} cuda={torch.cuda.is_available()}")
for path in ("/kaggle/working", "/kaggle/temp"):
    total, used, free = shutil.disk_usage(path)
    print(f"{path}: total={total/1e9:.1f}GB free={free/1e9:.1f}GB")

In [ ]:
!pip install -q uv
!uv --version

In [ ]:
# Copy code repo tu /kaggle/input sang /kaggle/working (input la read-only)
from pathlib import Path
import shutil

REPO = Path("/kaggle/working/IR")
if not REPO.exists():
    candidates = [
        p.parent
        for p in Path("/kaggle/input").rglob("pyproject.toml")
        if (p.parent / "src" / "ir").exists()
    ]
    if not candidates:
        raise FileNotFoundError(
            "Khong tim thay repo. Attach code nhu Kaggle Dataset hoac git clone vao /kaggle/working/IR"
        )
    shutil.copytree(
        candidates[0],
        REPO,
        ignore=shutil.ignore_patterns(".git", "artifacts", "node_modules", "*.mp4"),
    )
    print("Copied code from", candidates[0])
else:
    print("Repo da ton tai:", REPO)

In [ ]:
%cd /kaggle/working/IR
!uv pip install --system -e .
!uv pip install --system --group ml --group index
import av, torch, transnetv2_pytorch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Quet video trong TAT CA dataset dang attach (/kaggle/input), ho tro nhieu batch
from pathlib import Path

VIDEO_EXTS = {".mp4", ".mkv", ".avi", ".webm", ".mov", ".m4v"}
videos = sorted(
    p
    for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS
)
assert videos, "Chua attach dataset video nao vao /kaggle/input"
roots = sorted({p.parent for p in videos})
TEST_CORPUS = videos[0].parent
print(f"videos={len(videos)} folders={len(roots)}")
print("vi du:", roots[0])

In [ ]:
# Restore artifact tu lan chay truoc (Add Input -> Your Work -> Notebooks -> version cu)
from pathlib import Path
import shutil

previous = list(Path("/kaggle/input").rglob("metadata.sqlite"))
if previous:
    src = max(previous, key=lambda p: p.stat().st_mtime).parent
    shutil.copytree(src, REPO / "artifacts", dirs_exist_ok=True)
    print("Restored artifacts from", src)
else:
    print("Khong co artifact cu, bat dau index moi")

In [ ]:
# Chay thu 1 video truoc khi index hang loat
!ir index --corpus "{TEST_CORPUS}" --limit 1
!ir stats

In [ ]:
# Chia corpus thanh shard nho de vua 12h/session
from pathlib import Path
import shutil

SHARD_SIZE = 5
SHARDS_ROOT = Path("/kaggle/temp/shards")
shutil.rmtree(SHARDS_ROOT, ignore_errors=True)
shards = []
for index in range(0, len(videos), SHARD_SIZE):
    shard = SHARDS_ROOT / f"shard_{index // SHARD_SIZE:03d}"
    shard.mkdir(parents=True, exist_ok=True)
    for video in videos[index : index + SHARD_SIZE]:
        (shard / video.name).symlink_to(video)
    shards.append(shard)
print(f"{len(shards)} shards, {SHARD_SIZE} videos/shard")

In [ ]:
# Index tung shard, dung truoc 12h de kip Save Version
import subprocess, sys, time

DEADLINE_HOURS = 11.0
deadline = time.time() + DEADLINE_HOURS * 3600
for shard in shards:
    if time.time() >= deadline:
        print("Het thoi gian, dung de Save Version va resume o session sau")
        break
    print(f"\n=== {shard.name} ===", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "ir.cli", "index", "--corpus", str(shard)]
    )
    if result.returncode != 0:
        print(f"{shard.name} co video loi, xem status trong DB", flush=True)
print("Ket thuc phan chay trong session nay")

In [ ]:
# Kiem tra ket qua + dung luong artifact (gioi han output ~20GB)
import shutil

!ir stats
artifacts = REPO / "artifacts"
size = sum(p.stat().st_size for p in artifacts.rglob("*") if p.is_file())
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"artifacts={size/1e9:.2f} GB | working_free={free/1e9:.1f} GB")
if size > 18e9:
    print("CANH BAO: artifact gan/vuot 20GB output. Hay chia nho hoac luu artifact ra ngoai.")

## Resume qua nhieu session

1. Chay xong mot phan -> **Save Version** (hoac dung **Save & Run All** de chay nen).
2. Session moi: **Add Input -> Your Work -> Notebooks -> chon version cu**.
3. Chay lai cell **Restore artifact**; cell nay tim `metadata.sqlite` trong `/kaggle/input` va copy vao `/kaggle/working/IR/artifacts`.
4. Chay lai cell index; video da `done` se duoc skip.

## Neu corpus khong nam tren Kaggle Dataset

Tai tung shard vao `/kaggle/temp` (khong tinh vao output 20GB), vi du:

```bash
pip install -q gdown
gdown <DRIVE_FILE_ID> -O /kaggle/temp/shard.zip
unzip -q /kaggle/temp/shard.zip -d /kaggle/temp/corpus
```

Sau do sua cell quet video thanh `videos = sorted(Path("/kaggle/temp/corpus").rglob("*.mp4"))`, chay lai cell chia shard roi moi index.

## Ghi chu

- Artifact 100GB corpus uoc tinh ~8-10GB (keyframe 448px ~46MB/gio video), nam trong gioi han 20GB.
- Phase 2 can them group `serve` (FAISS + FastAPI): `uv pip install --system --group ml --group index --group serve`.
- Log loi cua tung video nam trong bang `index_status` (cot `error`).